# Correction-factor maps by calibration level: leave-one-out, noisy vs denoised scans

Throw-away copy of the `b800_corr_map_framework_comparison` figure from `figures.ipynb` with two changes:

1. **Noisy or denoised scans** (`SCAN_TYPES`): `RAW_REG` (registered noisy) or `RAW_MPPCA_REG` (registered MP-PCA denoised).
2. **Independent train and test data.** The scanner- and duration-level predictions are no longer made with calibration
   maps fitted to all repeats (`calibration/calib/*.nii.gz`). Instead `cal.loo_bmap_predictions` holds out repeat *k* of every
   sequence, fits the calibration to the other repeats, and predicts the held-out scans. The sequence-level prediction
   (the average of the other repeats) was already leave-one-out.

The existing figure corresponds to denoised scans with the in-sample maps.
Panels from `figures.ipynb` that don't feed this figure are not copied.

In [ ]:
# General libraries
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path
from functools import partial

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Custom functions
import registration as reg
import temperature as temp
import calib_utils as cal

In [ ]:
SCAN_TYPES = ["denoised", "noisy"]    # datasets to run
SCAN_FILES = {"noisy": "RAW_REG",          # registered, noisy scans
              "denoised": "RAW_MPPCA_REG"} # registered, MP-PCA denoised scans

b0_threshold = 5
B_EVAL = 700        # scans with b >= B_EVAL are evaluated (the b=800 scans)
SHRINK_DIST = 6     # mm, inset of the calibration mask from the threshold based mask

# Define directories
parent_dir = Path("./calibration")
deriv_dir = parent_dir / "derivatives"
plot_dir = parent_dir / "plots"
plot_dir.mkdir(exist_ok=True)
id_name = parent_dir.stem

param_fname = deriv_dir / (id_name + "_" + "params.h5")
mask_fname = deriv_dir / (id_name + "_" + "MASK.nii.gz")

### LOAD DATA AND CALCULATE B-MAPS

In [ ]:
def build_dwi_df(scan_type):
    """
    Load the scans, calculate the b-maps, and the smoothed gradient maps used to fit the calibration.
    Follows figures.ipynb (cells 2-3) and calibration_mapping.ipynb.

    :return: dwi_df (one row per diffusion-weighted scan), calib_mask
    """

    scans_fname = deriv_dir / (id_name + "_" + SCAN_FILES[scan_type] + ".nii.gz")

    # Unpack data from files
    scans_nii = nib.load(scans_fname)
    scans_data = scans_nii.get_fdata().astype(np.float32)
    scan_aff = scans_nii.affine
    scan_spacing = reg.spacing_from_affine(scan_aff)

    df = pd.read_hdf(param_fname, key="df", mode="r")
    scans_reshuffled = np.transpose(scans_data, [3,0,1,2])
    df["image"] = list(scans_reshuffled)
    df["adc"] = df.apply(lambda row: temp.water_diffusivity(temp=row["temperature"], polynomial=True), axis=1)

    preprocess_mask = nib.load(mask_fname).get_fdata().astype(bool)
    calib_mask = cal.shrink_mask_mm(preprocess_mask, scan_spacing, margin_mm=SHRINK_DIST)

    df.sort_values("scan_time", inplace=True, ignore_index=True)

    # Put the b=0 image on b=0 rows, NaN elsewhere
    df["s0_image"] = df["image"].where(df["b_value"] < b0_threshold)

    # Forward-fill so each DWI scan gets the most recent preceding b=0 image
    df["s0_image"] = df["s0_image"].ffill()

    df["bmap"] = df.apply(lambda row: cal.bmap_3d(s0=row["s0_image"], 
                                                  s=row["image"], 
                                                  adc=row["adc"],
                                                  mask=preprocess_mask), 
                                                  axis=1)

    # Keep only diffusion-weighted scans
    dwi_df = df[df["b_value"] > b0_threshold].copy().reset_index(drop=True)
    dwi_df["cmap_meas"] = dwi_df["bmap"]/dwi_df["b_value"]
    dwi_df = dwi_df.drop(columns=["image", "s0_image"])

    # Estimate gradient maps from b-value maps assuming Stejskal-Tanner Equation applies,
    # and a smoothed version of each for fitting the calibration (as calibration_mapping.ipynb)
    dwi_df["gmap2"] = dwi_df.apply(lambda row: cal.b2grad2(b=row["bmap"],
                                                           delta_small=row["dwi_grad_duration"],
                                                           delta_big=(row["TE"] / 2),
                                                           mask=calib_mask),
                                                           axis=1)

    dwi_df["gmap2_bspline"] = dwi_df.apply(lambda row: cal.bspline_estimation(data=row["gmap2"], spacing=row["spacing"], mask=calib_mask), axis=1)
    dwi_df = dwi_df.drop(columns=["gmap2"])

    return dwi_df, calib_mask

### FIGURE

In [ ]:
def plot_correction_factor_kdes(plot_df, calib_mask, fname, title=None):
    """
    Distribution of the correction factor (measured b / predicted b) in the calibration mask, for each
    diffusion direction (rows) and calibration level (columns), coloured by gradient duration.
    Copied from figures.ipynb.
    """

    plot_df = plot_df.sort_values(by="scan_time", ignore_index=True)
    dwi_dur = np.sort(plot_df["dwi_grad_duration"].unique())

    n_row = 3
    n_col = 4

    fig, axes = plt.subplots(
        n_row,
        n_col,
        figsize=(19, 9),
        constrained_layout=True,
        gridspec_kw={"width_ratios": [1,1,1,1]},
    )

    # Define font sizes in figure
    fs_plottitle = 16
    fs_axistitle = 10
    fs_legend = 12

    # Define colour maps
    rocket = sns.color_palette("rocket", 13)
    palette = [rocket[i+1] for i in range(12)]

    v_lims = (0.6, 1.4)
    cmap_list = ["cmap_meas", "cmap_resid_full", "cmap_resid_dur", "cmap_resid_seqspec"]
    column_labels = ["No calibration", "Scanner-level calibration", "Duration-level calibration", "Sequence-level calibration"]

    for i, dw_ax in enumerate("XYZ"):

        for j, dur in enumerate(dwi_dur):

            sub_df = plot_df[(plot_df["dwi_direction"]==dw_ax) & (plot_df["dwi_grad_duration"]==dur)].copy()

            for k, cmap_label in enumerate(cmap_list):

                cmap = np.stack(sub_df[cmap_label].values, axis=-1)

                for l, c in enumerate(cmap.transpose(3,0,1,2)):

                    kde_vals = c[calib_mask].flatten()
                    if l == 0:
                        sns.kdeplot(kde_vals, ax=axes[i, k], color=palette[j+3*k], lw=1, clip=v_lims, alpha=0.5+0.5*j/2, fill=False, label=rf"$\delta = {dur*1e3:.0f}\ ms$")
                    else:
                        sns.kdeplot(kde_vals, ax=axes[i, k], color=palette[j+3*k], lw=1, clip=v_lims, alpha=0.5+0.5*j/2, fill=False, legend=False)
        
                axes[i, k].set_xlim(v_lims)
                axes[i, k].set_ylabel("Density")
                axes[i, k].grid(alpha=0.25)
                axes[i, k].spines["top"].set_visible(False)
                axes[i, k].spines["right"].set_visible(False)
                axes[i, k].set_ylabel(None)
                axes[i, k].set_ylim(0, 10)
                
            
                if i==0:
                    axes[i, k].set_title(column_labels[k], fontsize=fs_plottitle, fontstyle="italic", pad=12)
                    axes[i, k].legend(fontsize=fs_legend, frameon=False)
                elif i==2:
                    axes[i, k].set_xlabel(r"$Correction\ factor$", fontsize=fs_axistitle)

        axes[i, 0].text(
            -0.1, 0.5,
            rf"${dw_ax}$",
            transform=axes[i, 0].transAxes,
            fontsize=fs_plottitle,
            fontweight="normal",
            ha="right",
            va="center",
        )

    if title is not None:
        fig.suptitle(title, fontsize=fs_plottitle + 4)

    fig.savefig(fname=fname, dpi=600, format="png")
    plt.show()

### RUN: LEAVE-ONE-OUT PREDICTIONS AND FIGURES

In [ ]:
metric_rows = []

for scan_type in SCAN_TYPES:

    print(f"==== {scan_type} scans ====")
    dwi_df, calib_mask = build_dwi_df(scan_type)

    # Leave-one-repeat-out predictions for the evaluated (b=800) scans.
    # Every scan is used for training (all b-values) - only the held-out repeat is excluded.
    pred_df = cal.loo_bmap_predictions(dwi_df, calib_mask, eval_mask=dwi_df["b_value"] >= B_EVAL)

    plot_df = dwi_df.loc[pred_df.index].copy()
    plot_df["cmap_resid_full"] = None
    plot_df["cmap_resid_dur"] = None
    plot_df["cmap_resid_seqspec"] = None

    for idx, row in plot_df.iterrows():

        # as figures.ipynb: scanner- and duration-level maps are valid in the calibration mask,
        # the sequence-level map in a mask of the averaged b-map
        for level, col in [("scanner", "cmap_resid_full"), ("duration", "cmap_resid_dur"), ("sequence", "cmap_resid_seqspec")]:

            pred_bmap = pred_df.at[idx, f"pred_bmap_{level}"]
            mask = reg.ants_mask(pred_bmap, fmt=bool) if level == "sequence" else calib_mask

            cmap_resid = np.zeros_like(mask, dtype=np.float32)
            cmap_resid[mask] = row["bmap"][mask] / pred_bmap[mask]
            plot_df.at[idx, col] = cmap_resid

    # Spread of the correction factor in the calibration mask (a perfect correction would be 1 everywhere)
    for label, col in [("No calibration", "cmap_meas"), ("Scanner", "cmap_resid_full"),
                       ("Duration", "cmap_resid_dur"), ("Sequence", "cmap_resid_seqspec")]:
        vals = np.concatenate([c[calib_mask].ravel() for c in plot_df[col].values])
        vals = vals[(vals > 0.6) & (vals < 1.4)] # the range plotted
        metric_rows.append({"scan_type": scan_type, "level": label,
                            "median": np.median(vals), "std": vals.std(),
                            "iqr": np.percentile(vals, 75) - np.percentile(vals, 25),
                            "iwr_5_95": np.percentile(vals, 95) - np.percentile(vals, 5)})

    plot_fname = plot_dir / (id_name + "_" + f"b800_corr_map_framework_comparison_LOO_{scan_type}.png")
    plot_correction_factor_kdes(plot_df, calib_mask, plot_fname, title=f"{scan_type} scans, leave-one-out")

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(plot_dir / (id_name + "_" + "b800_corr_map_framework_comparison_LOO_metrics.csv"), index=False)
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))